In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [15]:
df = pd.read_csv("Housing.csv")
print("\nFIRST FIVE RECORDS") 
print(df.head()) 
print("\nDATASET SHAPE") 
print(df.shape) 
print("\nCOLUMN NAMES") 
print(df.columns.tolist()) 
print("\nDATA TYPES AND NON-NULL COUNTS") 
df.info() 
print("\nMISSING VALUES") 
print(df.isnull().sum()) 
print("\nDUPLICATE ROWS:", df.duplicated().sum()) 
print("\nDESCRIPTIVE STATISTICS") 
print(df.describe().T)


FIRST FIVE RECORDS
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  

DATASET SHAPE
(545, 13)

COLUMN NAMES
['price', 'area', 'bedrooms', 'bathrooms

In [16]:
# 2. Handle Categorical / Binary Variables
# Map binary 'yes'/'no' columns to 1 and 0
binary_vars = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for var in binary_vars:
    df[var] = df[var].map({'yes': 1, 'no': 0})

# Encode multi-category variables (like 'furnishingstatus') using One-Hot Encoding
df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)

# 3. Define Independent (X) and Dependent (y) variables
X = df.drop(columns=['price'])
y = df['price']

# 4. Split the data into training (70%) and test (30%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 5. Scale the training and test data using StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both training and test sets to prevent data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train scaled shape: {X_train_scaled.shape}")
print(f"X_test scaled shape: {X_test_scaled.shape}")

X_train scaled shape: (381, 13)
X_test scaled shape: (164, 13)


In [17]:
# 3. Split the dataset into train (70%) and test (30%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Apply and Fit Ridge Regression
ridge = Ridge(alpha=1.0) # alpha is the regularization strength parameter
ridge.fit(X_train_scaled, y_train)
ridge_pred = ridge.predict(X_test_scaled)

# 6. Apply and Fit Lasso Regression
lasso = Lasso(alpha=1.0, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
lasso_pred = lasso.predict(X_test_scaled)

# 7. Evaluate both models
print("--- Ridge Regression Evaluation ---")
print(f"MAE: {mean_absolute_error(y_test, ridge_pred):.2f}")
print(f"MSE: {mean_squared_error(y_test, ridge_pred):.2f}")
print(f"R2-Score: {r2_score(y_test, ridge_pred):.4f}\n")

print("--- Lasso Regression Evaluation ---")
print(f"MAE: {mean_absolute_error(y_test, lasso_pred):.2f}")
print(f"MSE: {mean_squared_error(y_test, lasso_pred):.2f}")
print(f"R2-Score: {r2_score(y_test, lasso_pred):.4f}")

--- Ridge Regression Evaluation ---
MAE: 920170.55
MSE: 1522881286264.39
R2-Score: 0.6464

--- Lasso Regression Evaluation ---
MAE: 920392.75
MSE: 1523019776489.35
R2-Score: 0.6463


In [18]:
new_house = pd.DataFrame({
    'area': [5000],
    'bedrooms': [3],
    'bathrooms': [2],
    'stories': [2],
    'mainroad': ['yes'],
    'guestroom': ['no'],
    'basement': ['no'],
    'hotwaterheating': ['no'],
    'airconditioning': ['yes'],
    'parking': [2],
    'prefarea': ['yes'],
    'furnishingstatus': ['furnished']
})

In [19]:
new_house = pd.get_dummies(
    new_house,
    drop_first=True
)

In [20]:
new_house = new_house.reindex(
    columns=X.columns,
    fill_value=0
)

In [21]:
new_house_scaled = scaler.transform(new_house)
ridge_price = ridge.predict(new_house_scaled)

print("Predicted House Price using Ridge:", ridge_price[0])
lasso_price = lasso.predict(new_house_scaled)

print("Predicted House Price using Lasso:", lasso_price[0])

Predicted House Price using Ridge: 5272716.377181011
Predicted House Price using Lasso: 5275158.180922241
